# Extension a cuatro semanas (S1-S4) - ejecucion en Colab

Ventanas de 168 h: **S1** 26 abr-2 may, **S2** 3-9 may, **S3** 10-16 may, **S4** 17-23 may 2026 (semana original). Cada semana entrena y tunea solo con datos anteriores a su inicio (`windows.py` recorta los datos al fin de la ventana).
Plan completo: **216 corridas (54 por semana)**, ver `docs/plan_corridas_ventanas.csv`.

## Reparto PC / Colab (sin duplicados)
- **PC** (`scripts/run_local_estadisticos.py`): naive, naive_trend, naive_trend_seasonal, ar, ar_resid_trend_seasonal, sarimax, **xgboost y lightgbm** (T1 y T3) = 26 corridas por semana (104).
- **Colab (este cuaderno)**: las 4 redes (fcnn, lstm_resid, lstm_direct, ensemble_stl) en T2 y T3 = 28 corridas por semana (**112**). Esta celda solo lanza lo marcado `ejecuta_en = colab` en el plan.
- Resultados: `Pipeline_Resultados_Ventanas/<S#>/<RUN_NAME>/` (nunca `Pipeline_Resultados/`). `run_matrix` **salta** lo completo y **reanuda** lo incompleto region por region.

## Que ejecutar (las celdas de codigo se numeran 1 a 7, en orden)
1. Entorno de ejecucion: **GPU (T4)**.
2. **Celda 1 - Setup** (autoriza Drive). Si pide reiniciar, hazlo y vuelve a ejecutarla.
3. **Celda 3 - Progreso** (solo lectura): cuantas de las 112 estan completas y cuales faltan. **Celda 2 - Auditoria** es opcional (1 minuto).
4. **Celda 4 - Lanzamiento** con `SESION = "A"` y dejarla corriendo. En una segunda pestana, `SESION = "B"`. **Nunca la misma sesion en dos pestanas.** Si la RAM falla, usa lotes mas chicos (`LOTE`, ver la celda).
5. Mas tarde: **Celdas 5 y 6** (comparar S4 con los originales) y **Celda 7** (consolidado).

**Para reanudar** (corte, desconexion o falta de RAM): *Entorno de ejecucion > Reiniciar sesion*, celda 1 y celda 4 con la misma `SESION`. No hay que borrar nada ni repetir regiones: lo completo se salta y lo parcial continua.

## Celda 1 - Setup

Monta Drive, instala `optuna` y `lightgbm`, clona (o actualiza) el repositorio y verifica rutas, version y que el codigo de ejecucion aislada este presente. Si el `assert` de "Codigo desactualizado" falla, usa *Entorno de ejecucion > Reiniciar sesion* y vuelve a ejecutar esta celda (Python conservaba la version vieja del repositorio en memoria).

In [ ]:
import os
import sys
import subprocess

from google.colab import drive
drive.mount("/content/drive")

!pip install -q optuna lightgbm

REPO_URL = "https://github.com/CarlosT0503/Tesis-forecasting.git"
REPO_DIR = "/content/tesis_repo"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Rutas (deben coincidir con io_drive.py)
DATA_DIR = "/content/drive/MyDrive/Bases de datos Tesis"
ORIG_DIR = "/content/drive/MyDrive/Pipeline_Resultados"
VENT_DIR = "/content/drive/MyDrive/Pipeline_Resultados_Ventanas"
os.makedirs(os.path.join(VENT_DIR, "_auditoria"), exist_ok=True)

from tesis_forecast import io_drive, windows
from tesis_forecast import ventanas_tools as vt
from tesis_forecast.matrix import run_matrix, resumen_dataframe

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Version del repositorio:", commit)
assert io_drive.BASE_DIR_VENTANAS == VENT_DIR, "VENT_DIR no coincide con io_drive.BASE_DIR_VENTANAS"
assert io_drive.BASE_DIR == ORIG_DIR, "ORIG_DIR no coincide con io_drive.BASE_DIR"
assert hasattr(vt, "estado_plan") and "xgboost" in vt.MODELOS_LOCALES, "Codigo desactualizado: reinicia el entorno y vuelve a ejecutar esta celda"
from tesis_forecast import colab_aislado as ca, telemetria
assert hasattr(ca, "ejecutar_sesion") and hasattr(ca, "informe_progreso"), "Codigo desactualizado: Entorno de ejecucion > Reiniciar sesion y vuelve a ejecutar esta celda"
muestra = telemetria.snapshot()
print("Memoria al iniciar: RAM %.1f GB libres de %.1f | GPU %s de %s MB usados" % (
    muestra["ram_disponible_gb"], muestra["ram_total_gb"], muestra["gpu_usada_mb"], muestra["gpu_total_mb"]))
print("Ventanas:", {k: (str(v.test_start), str(v.test_end)) for k, v in windows.VENTANAS.items()})

## Celda 2 - Auditoria de datos (solo lectura, ~1 min)

Cobertura por region y semana, GEN/IMP/EXP y exogenas globales. Compara las huellas MD5 con `docs/datos_locales_MD5SUMS.txt`. Guarda CSV en `_auditoria/`; **no detiene** nada.

In [ ]:
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)


def seleccionar(df, columnas):
    """Solo las columnas que existen (la auditoria agrega algunas solo cuando hay problemas)."""
    return df[[c for c in columnas if c in df.columns]]


dem, ex, gl = vt.auditar_datos_ventanas(DATA_DIR)
for nombre, df in [("auditoria_demanda", dem), ("auditoria_exog_regional", ex), ("auditoria_exog_globales", gl)]:
    df.to_csv(os.path.join(VENT_DIR, "_auditoria", nombre + ".csv"), index=False, encoding="utf-8-sig")

print("== Regiones NO aptas para alguna semana (vacio = todas aptas):")
apta = dem["apta_para_ventana"].fillna(False).astype(bool) if "apta_para_ventana" in dem.columns else pd.Series(False, index=dem.index)
print(seleccionar(dem[~apta], ["ventana", "region", "ultima_fecha", "filas_ventana", "problema"]).to_string(index=False))

print("== Cobertura de la demanda (S4):")
print(seleccionar(dem[dem["ventana"] == "S4"], ["region", "archivo_fecha_min", "archivo_fecha_max", "horas_totales", "nan_demanda_previas", "horas_previas_validas", "apta_para_ventana"]).to_string(index=False))

print("== IMP/EXP/GEN: cobertura con valor real (S4). Si IMP/EXP terminan en 2019-12-31 son constantes por ffill:")
print(seleccionar(ex[ex["ventana"] == "S4"], ["region", "exog", "primera_fecha_con_valor", "ultima_fecha_con_valor", "pct_filas_con_valor"]).to_string(index=False))

print("== Exogenas globales (S4):")
print(gl[gl["ventana"] == "S4"].to_string(index=False))

plan = vt.plan_dataframe()
plan.to_csv(os.path.join(VENT_DIR, "_auditoria", "plan_corridas_ventanas.csv"), index=False, encoding="utf-8-sig")
print("== Plan:", len(plan), "corridas;", plan.groupby("ejecuta_en").size().to_dict())

!cd "{DATA_DIR}" && md5sum *_long.csv *_GEN.csv *_IMP.csv *_EXP.csv IGAE_2.xlsx "Temperaturas promedio.csv"

## Celda 3 - Progreso de las corridas de Colab (solo lectura)

Muestra cuantas de las **112** corridas asignadas a Colab estan completas, parciales o sin iniciar, por sesion (**A** = S4 y S1; **B** = S2 y S3), cuales faltan (`regiones_ok` = regiones ya guardadas de 8), los ultimos fallos de subproceso (con su clase: `ram_sistema`, `ram_python`, `gpu_memoria`, `error_codigo`, ...) y las ultimas muestras de memoria. Tambien avisa si alguna LSTM directa tiene un numero de intentos de Optuna distinto del presupuesto (10). Guarda la lista de faltantes en `_auditoria/faltan_colab.csv`. **No modifica ningun resultado**; puedes ejecutarla en otra pestana mientras corre el lanzamiento.

In [ ]:
inf = ca.informe_progreso(VENT_DIR)
ruta_faltan = ca.guardar_faltantes(VENT_DIR, inf["faltan"])
print("")
print("Lista completa de lo que falta (112 - completas): " + ruta_faltan)

## Celda 4 - LANZAMIENTO (dejar corriendo)

Cada configuracion se ejecuta en **su propio subproceso**: al terminar, el sistema operativo recupera toda su memoria; si un subproceso muere (RAM, GPU o error) se registra y la cola sigue con la siguiente. Orden: **T2 -> T3**, una semana tras otra.

- **Sesion A**: S4 y S1. **Sesion B**: S2 y S3 (otra pestana). **Nunca la misma sesion en dos pestanas.**
- **Lo completo se salta y lo parcial continua desde la ultima region guardada**: ningun resultado ya obtenido se repite ni se sobrescribe.
- **Reintentos acotados**: hasta `MAX_INTENTOS` por configuracion y se abandona si 2 intentos seguidos no agregan ninguna region. Si 3 configuraciones seguidas fallan por RAM, la cola se detiene (el entorno no alcanza: reinicia la sesion, usa RAM alta o un lote mas chico).
- **Lotes mas chicos** (`LOTE`): `T2` (8 corridas) -> `T3-fcnn`, `T3-lstm_resid`, `T3-ensemble_stl`, `T3-lstm_direct` (12 cada uno). Terminado un lote, *Entorno de ejecucion > Reiniciar sesion*, ejecuta la celda 1 y cambia `LOTE`.
- **Que ver despues**: celda 3 (progreso). Archivos en `Pipeline_Resultados_Ventanas/_auditoria/`: `subprocesos_colab_<A|B>.csv` (cada intento: clase de fallo, regiones antes/despues, pico de RAM), `telemetria_colab_<A|B>.csv` (RAM y GPU cada 2 min), `tiempos_colab_<A|B>.csv` y `faltan_colab.csv`.
- **Si se corta la sesion** (desconexion, 12 h): *Entorno de ejecucion > Reiniciar sesion*, celda 1 y celda 4 con el mismo `SESION` y `LOTE`.

In [ ]:
SESION = "A"        # "A": S4 y S1 (56 corridas)  |  "B": S2 y S3 (56 corridas)  -- una sesion por pestana
LOTE = "TODO"       # "TODO", o un lote mas chico: "T2", "T3-fcnn", "T3-lstm_resid", "T3-ensemble_stl", "T3-lstm_direct"
MAX_INTENTOS = 3    # por configuracion; cada reintento reanuda por region (no repite regiones completas)
assert SESION in ("A", "B"), "SESION debe ser 'A' o 'B'"

resumen, motivo = ca.ejecutar_sesion(SESION, LOTE, vent_dir=VENT_DIR, data_dir=DATA_DIR, max_intentos=MAX_INTENTOS)
if motivo:
    print("")
    print("LA COLA SE DETUVO: " + motivo)
resumen[resumen["status"] == "failed"][["modelo", "ventana", "run_name", "error"]] if len(resumen) else resumen

## Celdas 5 y 6 - Comparar la reproduccion de S4 con los resultados originales (solo lectura)

Ejecutar cuando termine S4. Las corridas hechas en la PC aparecen como `FALTA_REPRODUCCION` hasta que copies sus carpetas a Drive (ver `docs/INSTRUCCIONES_EJECUCION.md`). Naive, Naive_Trend y AR salen `NO_COMPARABLE` (su original usaba otra ventana).

Tolerancias propuestas: deterministicos 1e-6, arboles 1e-4, redes +-0.5 pp de MAPE por region.

In [ ]:
cfg_s4 = vt.construir_configs("T1", "S4") + vt.construir_configs("T2", "S4")
detalle, resumen_s4 = vt.comparar_s4_con_original(cfg_s4, ORIG_DIR, VENT_DIR)
resumen_s4

In [ ]:
detalle[~detalle["estado"].isin(["OK"])]

## Celda 7 - Consolidado: modelo x region x semana, con medias y desviaciones

Lee todo lo que haya en `Pipeline_Resultados_Ventanas/` (incluidos los resultados de la PC una vez copiados). Escribe en `Pipeline_Resultados_Ventanas/Consolidado/`.

In [ ]:
res = vt.consolidar_ventanas(VENT_DIR, out_dir=os.path.join(VENT_DIR, "Consolidado"))
print(res["inventario"].shape)
res["global_"].sort_values("MAPE_media").head(30)